# 18 - Unit Testing (`pytest`)


## 1. Unit Testing Fundamentals & Arrange-Act-Assert (AAA)

Birim testleri (unit tests), kod içindeki bağımsız fonksiyonların beklenen ve beklenmeyen girdiler karşısında doğru çalıştığını otomatik olarak doğrulayarak regresyon hatalarını (breaking changes) önler[cite: 92].

**Syntax**

```python
def test_addition() -> None:
    a, b = 1, 2
    actual = a + b
    assert actual == 3
```

**Temel Kavramlar ve Model**

* **Arrange – Act – Assert (AAA):** Test önce girdi ve ön koşulları hazırlar (*Arrange*), ardından test edilen fonksiyonu çağırır (*Act*) ve son olarak dönen sonucu beklentiyle `assert` üzerinden karşılaştırır (*Assert*).
* **`unittest` vs. `pytest`:** Python standart kütüphanesinde `unittest` yer alır; `pytest` ise sınıf kalıtımı gerektirmeden `test_` öneki veya `_test` soneki alan fonksiyonları otomatik keşfeder ve standart `assert` ifadeleriyle ayrıntılı hata dökümü sunar.
* **Sınır Durumları (Edge Cases):** Hatalı veri tipleri veya beklenmeyen girdiler için yazılan testler, fonksiyonun kırılgan noktalarını erken aşamada ortaya çıkarır[cite: 92].


In [ ]:
import pytest


def add_numbers(a: float, b: float) -> float:
    for value in (a, b):
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            return 0.0
    return float(a + b)


class TestAddNumbers:
    def test_valid_sum(self) -> None:
        a, b = 1, 2
        actual: float = add_numbers(a, b)
        expected: float = 3.0
        assert actual == expected

    def test_invalid_type_returns_zero(self) -> None:
        actual: float = add_numbers(1, 'z')
        assert actual == 0.0


suite = TestAddNumbers()
suite.test_valid_sum()
suite.test_invalid_type_returns_zero()
print('AAA tests passed.')


## 2. Reusable Test Setup with Fixtures (`@pytest.fixture`)

`@pytest.fixture` dekoratörü, birden fazla test fonksiyonunun ihtiyaç duyduğu ortak başlangıç verilerini, nesneleri veya hazırlık mantığını (*Arrange* aşamasını) tekrar etmeden testlere bağımlılık enjeksiyonu (dependency injection) yoluyla sağlar[cite: 93].

**Syntax**

```Python
@pytest.fixture
def sample_items() -> list[int]:
    return [-1, 3, 2, 1, 0]


def test_sorting(sample_items: list[int]) -> None:
    assert sorted(sample_items) == [-1, 0, 1, 2, 3]
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **İsim Eşleşmesi:** Test fonksiyonunun parametre adı fixture fonksiyonunun adıyla birebir aynı olduğunda `pytest` fixture'ı otomatik çalıştırır ve dönüş değerini parametreye aktarır.
* **İzolasyon:** Varsayılan `scope='function'` ayarında her test çağrısı için fixture yeniden çalıştırılır; böylece bir testin liste üzerinde yaptığı yerinde (`in-place`) değişiklik diğer testi bozmaz[cite: 93].


In [ ]:
import tempfile
from pathlib import Path

import pytest

with tempfile.TemporaryDirectory() as tmp_dir:
    root = Path(tmp_dir)
    (root / 'sample_module.py').write_text(
        'def sort_numbers(numbers: list[int]) -> list[int]:\n'
        '    numbers.sort()\n'
        '    return numbers\n\n'
        'def reverse_numbers(numbers: list[int]) -> list[int]:\n'
        '    numbers.reverse()\n'
        '    return numbers\n',
        encoding='utf-8',
    )
    (root / 'test_sample_module.py').write_text(
        'import pytest\n'
        'import sample_module as sm\n\n'
        '@pytest.fixture\n'
        'def sample_numbers_unsorted() -> list[int]:\n'
        '    return [-1, 3, 2, 1, 0]\n\n'
        'def test_sort_numbers(sample_numbers_unsorted: list[int]) -> None:\n'
        '    assert sm.sort_numbers(sample_numbers_unsorted) == [-1, 0, 1, 2, 3]\n\n'
        'def test_reverse_numbers(sample_numbers_unsorted: list[int]) -> None:\n'
        '    assert sm.reverse_numbers(sample_numbers_unsorted) == [0, 1, 2, 3, -1]\n',
        encoding='utf-8',
    )
    pytest.main(['-v', str(root)])


## 3. Shared Configuration (`conftest.py`), `autouse` & Monkeypatching

`conftest.py`, bulunduğu dizin ve alt dizinlerdeki tüm test modülleri tarafından `import` edilmeden otomatik olarak tanınan özel `pytest` yapılandırma dosyasıdır[cite: 94].

**Syntax**

```python
@pytest.fixture(autouse=True)
def mock_external_api(monkeypatch: pytest.MonkeyPatch) -> None:
    monkeypatch.setattr(target_module, 'fetch_api', lambda: 'mocked_response')
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **Otomatik Keşif:** `conftest.py` içinde tanımlanan fixture'lar aynı klasör ağacındaki tüm `test_*.py` dosyalarında doğrudan parametre olarak kullanılabilir.
* **`autouse=True`:** Test fonksiyonlarının parametre listesine eklenmesine gerek kalmadan her test çalışmadan önce otomatik devreye giren fixture'lar tanımlar.
* **Monkeypatching (Sahte Veri / Mock):** Canlı ağ istekleri, veritabanı çağrıları veya maliyetli servisler test sırasında `monkeypatch.setattr()` ile geçici olarak sahte fonksiyonlara yönlendirilir ve test bitiminde orijinal haline döndürülür[cite: 94].


In [ ]:
import tempfile
from pathlib import Path

import pytest

with tempfile.TemporaryDirectory() as tmp_dir:
    root = Path(tmp_dir)
    (root / 'service_module.py').write_text(
        'def reverse_text(text: str) -> str:\n'
        '    if not isinstance(text, str):\n'
        '        raise TypeError("Expected str")\n'
        '    return text[::-1]\n\n'
        'def get_api_data() -> str:\n'
        '    return "live_production_data"\n',
        encoding='utf-8',
    )
    (root / 'conftest.py').write_text(
        'import pytest\n'
        'import service_module as sm\n\n'
        '@pytest.fixture\n'
        'def sample_text() -> str:\n'
        '    return "banana"\n\n'
        '@pytest.fixture(autouse=True)\n'
        'def disable_live_api(monkeypatch: pytest.MonkeyPatch) -> None:\n'
        '    monkeypatch.setattr(sm, "get_api_data", lambda: "mocked_data")\n',
        encoding='utf-8',
    )
    (root / 'test_service.py').write_text(
        'import service_module as sm\n\n'
        'def test_api_returns_mock() -> None:\n'
        '    assert sm.get_api_data() == "mocked_data"\n\n'
        'def test_reverse_text(sample_text: str) -> None:\n'
        '    assert sm.reverse_text(sample_text) == "ananab"\n',
        encoding='utf-8',
    )
    pytest.main(['-v', str(root)])


## 4. Test Marks (`xfail`, `skip`, `skipif`)

`pytest.mark` dekoratörleri, test fonksiyonlarına üst veri (metadata) ekleyerek belirli testlerin koşulsuz atlanmasını, bir koşula bağlı olarak devre dışı bırakılmasını veya beklenen hataların raporlama davranışını yönetmeyi sağlar[cite: 95].

**Syntax**

```python
@pytest.mark.xfail(reason='Known bug #42')
@pytest.mark.skip(reason='Temporarily disabled')
@pytest.mark.skipif(sys.platform == 'win32', reason='POSIX only')
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **`@pytest.mark.xfail`:** Henüz çözülmemiş bilinen bir hata nedeniyle başarısız olması beklenen testi işaretler; test başarısız olduğunda `XFAIL` olarak raporlanır ve genel test paketini kırmaz.
* **`@pytest.mark.skip`:** Testi çalıştırmadan koşulsuz olarak atlar (`SKIPPED`).
* **`@pytest.mark.skipif(condition, reason=...)`:** İlk argümandaki boolean ifade `True` olarak değerlendirildiğinde testi belirtilen `reason` açıklamasıyla atlar; platform, Python sürümü veya ortam değişkeni kontrollerinde yaygın kullanılır[cite: 95].


In [ ]:
import tempfile
from pathlib import Path

import pytest

with tempfile.TemporaryDirectory() as tmp_dir:
    test_file = Path(tmp_dir) / 'test_marks.py'
    test_file.write_text(
        'import sys\n'
        'import pytest\n\n'
        '@pytest.mark.xfail(reason="Expected failure placeholder")\n'
        'def test_known_issue() -> None:\n'
        '    assert False\n\n'
        'def test_passing_case() -> None:\n'
        '    assert True\n\n'
        '@pytest.mark.skip(reason="Unconditional skip")\n'
        'def test_skipped_case() -> None:\n'
        '    assert 1 == 2\n\n'
        '@pytest.mark.skipif(sys.version_info >= (3, 10), reason="Legacy Python <3.10 only")\n'
        'def test_legacy_behavior() -> None:\n'
        '    assert True\n',
        encoding='utf-8',
    )
    pytest.main(['-v', '-rsx', str(test_file)])


## 5. Data-Driven Testing (`@pytest.mark.parametrize`)

`@pytest.mark.parametrize`, aynı test mantığını farklı girdi ve beklenen çıktı kombinasyonlarıyla otomatik olarak döngüye sokarak kod tekrarını önleyen veri odaklı test dekoratörüdür[cite: 96].

**Syntax**

```python
@pytest.mark.parametrize(
    'raw_input, expected',
    [
        ('abc', 'cba'),
        ('apple', 'elppa'),
        ('', ''),
    ],
)
def test_reverse(raw_input: str, expected: str) -> None:
    assert raw_input[::-1] == expected
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **Bağımsız Test Raporlaması:** Listedeki her bir öğe veya demet (tuple) `pytest` tarafından ayrı bir test vakası olarak çalıştırılır; bir girdinin başarısız olması diğer vakaların çalışmasını engellemez.
* **Tekli vs. Çoklu Parametre:** Tek parametre tanımlanırken değerler düz liste (`['a', 'b']`) olarak, birden fazla parametre virgülle ayrılmış string (`'arg1, arg2'`) olarak tanımlandığında ise değerler demet listesi (`[(x1, y1), (x2, y2)]`) olarak verilir[cite: 96].


In [ ]:
import tempfile
from pathlib import Path

import pytest

with tempfile.TemporaryDirectory() as tmp_dir:
    test_file = Path(tmp_dir) / 'test_parametrize.py'
    test_file.write_text(
        'import pytest\n\n'
        '@pytest.mark.parametrize(\n'
        '    "text_to_reverse, expected_result",\n'
        '    [\n'
        '        ("abc", "cba"),\n'
        '        ("apple", "elppa"),\n'
        '        ("", ""),\n'
        '    ],\n'
        ')\n'
        'def test_reverse_text(text_to_reverse: str, expected_result: str) -> None:\n'
        '    assert text_to_reverse[::-1] == expected_result\n\n'
        '@pytest.mark.parametrize("sample_item", ["apple", "banana", "orange"])\n'
        'def test_item_is_str(sample_item: str) -> None:\n'
        '    assert isinstance(sample_item, str)\n',
        encoding='utf-8',
    )
    pytest.main(['-v', str(test_file)])


## 6. Testing Expected Exceptions (`pytest.raises`)

Bir fonksiyonun hatalı girdiler veya geçersiz durumlar karşısında belirli bir istisna (exception) fırlattığını doğrulamak için `with pytest.raises(...)` bağlam yöneticisi kullanılır[cite: 97].

**Syntax**

```python
with pytest.raises(TypeError, match='Expected str'):
    target_function(invalid_arg)
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **Tam İstisna Eşleşmesi:** `with` bloğu içinde belirtilen istisna fırlatılırsa test başarılı olur; hiçbir istisna fırlatılmazsa veya farklı bir istisna türü fırlatılırsa test başarısız sayılır.
* **Hata Mesajı Doğrulama (`match`):** İsteğe bağlı `match` parametresi ile fırlatılan istisnanın mesajının belirli bir düzenli ifadeyle (regex) eşleşip eşleşmediği de kontrol edilebilir[cite: 97].


In [ ]:
import tempfile
from pathlib import Path

import pytest

with tempfile.TemporaryDirectory() as tmp_dir:
    test_file = Path(tmp_dir) / 'test_exceptions.py'
    test_file.write_text(
        'import pytest\n\n'
        'def ensure_text(value: str) -> str:\n'
        '    if not isinstance(value, str):\n'
        '        raise TypeError("Input must be a string")\n'
        '    return value\n\n'
        'def test_int_raises_type_error() -> None:\n'
        '    with pytest.raises(TypeError, match="must be a string"):\n'
        '        ensure_text(9)\n\n'
        'def test_valid_string_succeeds() -> None:\n'
        '    assert ensure_text("ok") == "ok"\n',
        encoding='utf-8',
    )
    pytest.main(['-v', str(test_file)])
